# Music Annotation Project V7.9: Observed-Ink / Glyph-Preservation Challenger

**Notebook 22. Runtime:** CPU. Select **Runtime > Run all**.

This corrected version performs a memory-bounded two-pass benchmark:

1. **Streamed candidate search:** precompute page-invariant evidence once, score each candidate, retain only scalar rows, and discard candidate arrays immediately.
2. **Finalist diagnostics:** reconstruct only frozen V7.8 plus the selected finalists and create diagnostic crops.

## Hard guardrails

- Notebook 09 is not read or modified.
- Frozen V7.8 independent input, outcome and policy-validation directories are prohibited.
- Only controlled development pages are used.
- Clean pages are introduced only after candidate decisions are formed, for scoring.
- Dilation 2, Telea, paper estimation, V7.8 supports, remove semantics and ambiguous semantics remain frozen.
- Pixels outside the removal mask must remain byte-for-byte unchanged.
- Frozen V7.8 must reproduce the durable V7.5 cache exactly before challenger search.
- No V7.7F probability is applied to changed V7.9 routing semantics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib psutil


## Configuration and fail-closed development boundary


In [ ]:
import cv2, json, hashlib, gc, math, os, psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9'
V75_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
MAX_PAGES=None

MASK_DILATION=2
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4

DARK_THRESHOLDS=[80,100,120,140,160]
STROKE_WIDTH_LIMITS=[2.5,3.5,5.0]
MIN_GLYPH_COMPONENT_AREAS=[3,6,12]
MAX_GLYPH_COMPONENT_AREAS=[650,1200,2400]
MODES=['connected_dark','edge_stroke','consensus','union']
FINALIST_COUNT=3
MAX_DIAGNOSTIC_COMPONENTS=30

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
PROHIBITED_PATH_TOKENS=[
 'independent_structural_validation_v7_8_builder',
 'independent_structural_validation_v7_8_rebuilt',
 'independent_policy_validation_v7_8',
]

def assert_development_path(path):
    text=str(Path(path)).lower()
    if any(token in text for token in PROHIBITED_PATH_TOKENS):
        raise RuntimeError(f'PROHIBITED V7.8 independent path accessed: {path}')

def latest_complete(root,required):
    assert_development_path(root);hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def rss_gb():return psutil.Process(os.getpid()).memory_info().rss/(1024**3)
def memory_row(stage,page=None,candidate=None):return {'utc':datetime.now(timezone.utc).isoformat(),'stage':stage,'page':page,'candidate':candidate,'rss_gb':rss_gb()}

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv'])
if V75 is None:raise RuntimeError('No complete controlled V7.5 development run found.')
assert_development_path(V75)
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';CHARTS=OUT/'charts';CONTACTS=OUT/'contact_sheets';FINALISTS=OUT/'finalists'
for p in [OUT,TABLES,CHARTS,CONTACTS,FINALISTS]:p.mkdir(parents=True,exist_ok=False)
print('Controlled V7.5 reference:',V75);print('Output:',OUT);print('Initial RSS GB:',round(rss_gb(),3))


## Frozen V7.8 and challenger functions


In [ ]:
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma)
    surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps_v78(observed,mask):
    dark=(observed<190)&(mask==0)
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if 3<=a<=650 and w<=40 and h<=40:compact[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
    return cv2.dilate(horizontal.astype(np.uint8),k)>0,cv2.dilate(vertical.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0

def page_invariants(observed,mask,telea,clean):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255)
    horizontal,vertical,compact=support_maps_v78(observed,mask)
    edges_observed=cv2.Canny(observed,70,170)>0
    clean_ink=cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
    clean_edges=cv2.Canny(clean,70,170)>0
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    components=[(cid,labels==cid) for cid in range(1,n) if stats[cid,cv2.CC_STAT_AREA]>=MIN_COMPONENT_AREA]
    return {'paper':paper,'normalised':normalised,'horizontal':horizontal,'vertical':vertical,'compact':compact,'edges_observed':edges_observed,'clean_ink':clean_ink,'clean_edges':clean_edges,'components':components,'telea':telea}

def baseline_routes(observed,mask,inv):
    m=mask>0;preserve=m&(inv['horizontal']|inv['vertical']|inv['compact'])&(observed<200)
    remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(inv['normalised']>=GEOMETRY_THRESHOLD);ambiguous=m&~preserve&~remove
    return preserve,remove,ambiguous

def render(observed,mask,inv,preserve,remove,ambiguous):
    result=observed.copy();result[remove]=inv['paper'][remove];result[ambiguous]=inv['telea'][ambiguous];result[preserve]=observed[preserve]
    return immutable(observed,result,mask)

def boundary_connected_components(dark,mask,min_area,max_area):
    boundary=(mask>0)&~(cv2.erode((mask>0).astype(np.uint8),np.ones((7,7),np.uint8))>0)
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);out=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i];comp=lab==i
        if min_area<=a<=max_area and w<=80 and h<=80 and np.any(comp&boundary):out[comp]=True
    return out

def candidate_glyph(observed,mask,inv,params,mode):
    dark=(observed<params['dark_threshold'])&(mask>0)
    connected=boundary_connected_components(dark,mask,params['min_area'],params['max_area'])
    width=cv2.distanceTransform(dark.astype(np.uint8),cv2.DIST_L2,5)
    edge_stroke=dark&(cv2.dilate(inv['edges_observed'].astype(np.uint8),np.ones((3,3),np.uint8))>0)&(width<=params['stroke_width_limit'])
    raw={'connected_dark':connected,'edge_stroke':edge_stroke,'consensus':connected&edge_stroke,'union':connected|edge_stroke}[mode]
    n,lab,stats,_=cv2.connectedComponentsWithStats(raw.astype(np.uint8),8);clean=np.zeros_like(raw)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]>=params['min_area']:clean[lab==i]=True
    return clean

def candidate_routes(observed,mask,inv,params=None,mode=None):
    glyph=np.zeros_like(mask,dtype=bool) if mode is None else candidate_glyph(observed,mask,inv,params,mode)
    m=mask>0;preserve=m&(inv['horizontal']|inv['vertical']|inv['compact']|glyph)&(observed<200)
    remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(inv['normalised']>=GEOMETRY_THRESHOLD);ambiguous=m&~preserve&~remove
    return preserve,remove,ambiguous,glyph


## Development roster and frozen-baseline equivalence


In [ ]:
def image_map(folder):
    assert_development_path(folder)
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
    assert_development_path(path)
    with np.load(path) as z:return {k:z[k] for k in z.files}

clean_map=image_map(ROOT/'source'/'clean');observed_map=image_map(ROOT/'aligned')
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
pages=sorted(set(clean_map)&set(observed_map)&set(cache_lookup))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
if not pages:raise RuntimeError('Controlled development roster is empty.')
checks=[];memory_log=[memory_row('start')]
for page in pages:
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0)
    for key in ['mask','telea','geometry_t80']:
        if key not in z:raise RuntimeError(f'{page}: missing {key}')
    inv=page_invariants(observed,z['mask'],z['telea'],clean);p,r,a=baseline_routes(observed,z['mask'],inv);rebuilt=render(observed,z['mask'],inv,p,r,a)
    checks.append({'page':page,'geometry_exact':bool(np.array_equal(rebuilt,z['geometry_t80'])),'different_pixels':int(np.sum(rebuilt!=z['geometry_t80'])),'outside_immutable':bool(np.array_equal(rebuilt[z['mask']==0],observed[z['mask']==0]))})
    del z,clean,observed,inv,p,r,a,rebuilt;gc.collect()
baseline_df=pd.DataFrame(checks);baseline_df.to_csv(TABLES/'baseline_equivalence.csv',index=False)
if not baseline_df.geometry_exact.all() or not baseline_df.outside_immutable.all():raise RuntimeError('Frozen V7.8 baseline equivalence failed.')
print('Baseline reproduced on',len(pages),'pages. RSS GB:',round(rss_gb(),3))


## Streamed candidate search

No full candidate reconstruction or routing dictionary is retained. Each candidate is rendered, scored and discarded. CSV checkpoints are written after every page.


In [ ]:
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.sum():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))

def score_arrays(clean,recon,mask,inv,preserve,glyph,remove):
    m=mask>0;recon_ink=cv2.adaptiveThreshold(recon,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0;recon_edges=cv2.Canny(recon,70,170)>0
    ip,ir,if1=tolerant_scores(inv['clean_ink'],recon_ink,m);ep,er,ef1=tolerant_scores(inv['clean_edges'],recon_edges,m)
    diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));false_preserved=int(np.sum(preserve&~inv['clean_ink']));true_preserved=int(np.sum(preserve&inv['clean_ink']))
    return {'mask_pixels':int(m.sum()),'mae':float(diff[m].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'preserve_pixels':int(preserve.sum()),'true_print_preserved':true_preserved,'false_annotation_preserved':false_preserved,'preserve_precision':true_preserved/max(int(preserve.sum()),1),'glyph_pixels':int(glyph.sum()),'glyph_true_print':int(np.sum(glyph&inv['clean_ink'])),'glyph_false_annotation':int(np.sum(glyph&~inv['clean_ink'])),'removed_clean_print_fraction':float(np.sum(remove&inv['clean_ink']&m)/max(m.sum(),1)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m]))}

configs=[{'dark_threshold':dt,'stroke_width_limit':sw,'min_area':mina,'max_area':maxa} for dt in DARK_THRESHOLDS for sw in STROKE_WIDTH_LIMITS for mina in MIN_GLYPH_COMPONENT_AREAS for maxa in MAX_GLYPH_COMPONENT_AREAS if mina<=maxa]
rows=[]
for pi,page in enumerate(pages,1):
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean)
    preserve,remove,ambiguous,glyph=candidate_routes(observed,mask,inv);recon=render(observed,mask,inv,preserve,remove,ambiguous);rows.append({'page':page,'candidate':'v78_baseline',**score_arrays(clean,recon,mask,inv,preserve,glyph,remove)})
    del preserve,remove,ambiguous,glyph,recon
    for mode in MODES:
        for params in configs:
            name=f"v79_{mode}_d{params['dark_threshold']}_w{params['stroke_width_limit']}_a{params['min_area']}_{params['max_area']}"
            preserve,remove,ambiguous,glyph=candidate_routes(observed,mask,inv,params,mode);recon=render(observed,mask,inv,preserve,remove,ambiguous)
            q=score_arrays(clean,recon,mask,inv,preserve,glyph,remove)
            if q['outside_mask_drift']!=0:raise RuntimeError(f'{page} {name}: outside-mask drift')
            rows.append({'page':page,'candidate':name,**q})
            del preserve,remove,ambiguous,glyph,recon,q
    pd.DataFrame(rows).to_csv(TABLES/'candidate_metrics_by_page_checkpoint.csv',index=False)
    memory_log.append(memory_row('page_complete',page))
    print(f'Page {pi}/{len(pages)} {page}: rows={len(rows)} RSS={rss_gb():.3f} GB')
    del z,clean,observed,mask,inv;gc.collect()
results=pd.DataFrame(rows);results.to_csv(TABLES/'candidate_metrics_by_page.csv',index=False);pd.DataFrame(memory_log).to_csv(TABLES/'memory_telemetry.csv',index=False)
print('Search complete. Candidates:',results.candidate.nunique(),'Peak sampled RSS GB:',round(pd.DataFrame(memory_log).rss_gb.max(),3))


## Rank and select development finalists


In [ ]:
summary=results.groupby('candidate',as_index=False).agg(pages=('page','nunique'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),mae=('mae','mean'),preserve_precision=('preserve_precision','mean'),glyph_precision=('glyph_precision','mean'),false_annotation_preserved=('false_annotation_preserved','sum'),true_print_preserved=('true_print_preserved','sum'),removed_clean_print_fraction=('removed_clean_print_fraction','mean'))
base=summary[summary.candidate=='v78_baseline'].iloc[0]
summary['delta_ink_recall']=summary.ink_recall-base.ink_recall;summary['delta_edge_f1']=summary.edge_f1-base.edge_f1;summary['delta_mae']=summary.mae-base.mae;summary['delta_false_annotation_preserved']=summary.false_annotation_preserved-base.false_annotation_preserved
summary['false_preservation_rate']=summary.false_annotation_preserved/(summary.true_print_preserved+summary.false_annotation_preserved).clip(lower=1)
summary['utility']=summary.ink_recall.fillna(1)+summary.edge_f1.fillna(1)+summary.preserve_precision.fillna(1)-10*summary.false_preservation_rate-summary.removed_clean_print_fraction*20-summary.mae/255
eligible=summary[(summary.candidate!='v78_baseline')&(summary.delta_false_annotation_preserved<=0)&((summary.delta_ink_recall>0)|(summary.delta_edge_f1>0))].sort_values('utility',ascending=False)
finalists=eligible.head(FINALIST_COUNT).candidate.tolist()
summary.sort_values(['false_annotation_preserved','utility'],ascending=[True,False]).to_csv(TABLES/'candidate_summary.csv',index=False)
pd.DataFrame({'candidate':finalists,'rank':range(1,len(finalists)+1)}).to_csv(TABLES/'selected_finalists.csv',index=False)
print('Selected finalists:',finalists if finalists else 'NONE')
display(summary.sort_values(['false_annotation_preserved','utility'],ascending=[True,False]).head(30))


## Finalist-only reconstruction and component diagnostics

This second pass recreates only frozen V7.8 and the selected finalists. Full arrays are kept for one page at a time, diagnostic crops are written, then memory is released.


In [ ]:
def parse_candidate(name):
    import re
    m=re.fullmatch(r'v79_(connected_dark|edge_stroke|consensus|union)_d(\d+)_w([0-9.]+)_a(\d+)_(\d+)',name)
    if not m:raise ValueError(name)
    return m.group(1),{'dark_threshold':int(m.group(2)),'stroke_width_limit':float(m.group(3)),'min_area':int(m.group(4)),'max_area':int(m.group(5))}

def save_crop(page,cid,kind,rank,clean,observed,mask,outputs,labels,stats):
    x,y,w,h=stats[cid,:4];m=55;x0=max(0,x-m);y0=max(0,y-m);x1=min(clean.shape[1],x+w+m);y1=min(clean.shape[0],y+h+m)
    names=['clean truth','observed','frozen V7.8']+list(finalists);images=[clean,observed,outputs['v78_baseline']]+[outputs[n] for n in finalists]
    fig,axes=plt.subplots(1,len(images),figsize=(5*len(images),5),squeeze=False)
    for ax,im,name in zip(axes[0],images,names):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
    fig.suptitle(f'{kind} #{rank} | {page} component {cid}');plt.tight_layout();folder=CONTACTS/kind;folder.mkdir(parents=True,exist_ok=True);plt.savefig(folder/f'{rank:03d}_{page}_component_{cid:04d}.png',dpi=160,bbox_inches='tight');plt.close(fig)

component_rows=[]
for pi,page in enumerate(pages,1):
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean)
    outputs={};route_sets={}
    preserve,remove,ambiguous,glyph=candidate_routes(observed,mask,inv);outputs['v78_baseline']=render(observed,mask,inv,preserve,remove,ambiguous);route_sets['v78_baseline']=(preserve,remove,ambiguous,glyph)
    for name in finalists:
        mode,params=parse_candidate(name);preserve,remove,ambiguous,glyph=candidate_routes(observed,mask,inv,params,mode);outputs[name]=render(observed,mask,inv,preserve,remove,ambiguous);route_sets[name]=(preserve,remove,ambiguous,glyph)
        np.savez_compressed(FINALISTS/f'{page}_{name}.npz',reconstruction=outputs[name],preserve=preserve.astype(np.uint8),remove=remove.astype(np.uint8),ambiguous=ambiguous.astype(np.uint8),glyph=glyph.astype(np.uint8))
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for cid,comp in inv['components']:
        for name in ['v78_baseline']+finalists:
            recon=outputs[name];preserve,remove,ambiguous,glyph=route_sets[name];ip,ir,if1=tolerant_scores(inv['clean_ink'],cv2.adaptiveThreshold(recon,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0,comp);ep,er,ef1=tolerant_scores(inv['clean_edges'],cv2.Canny(recon,70,170)>0,comp)
            component_rows.append({'page':page,'component':cid,'candidate':name,'area':int(comp.sum()),'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'mae':float(np.abs(clean.astype(np.float32)-recon.astype(np.float32))[comp].mean()),'glyph_fraction':float(np.sum(glyph&comp)/comp.sum()),'preserve_fraction':float(np.sum(preserve&comp)/comp.sum()),'false_annotation_preserved':int(np.sum(preserve&comp&~inv['clean_ink']))})
    if finalists:
        cdf=pd.DataFrame([r for r in component_rows if r['page']==page]);pivot=cdf.pivot(index='component',columns='candidate',values=['ink_recall','false_annotation_preserved'])
        best=finalists[0];gain=(pivot['ink_recall'][best]-pivot['ink_recall']['v78_baseline']).sort_values(ascending=False).head(MAX_DIAGNOSTIC_COMPONENTS)
        harm=(pivot['false_annotation_preserved'][best]-pivot['false_annotation_preserved']['v78_baseline']).sort_values(ascending=False).head(MAX_DIAGNOSTIC_COMPONENTS)
        for rank,cid in enumerate(gain.index,1):save_crop(page,int(cid),'largest_ink_recall_gains',rank,clean,observed,mask,outputs,labels,stats)
        for rank,cid in enumerate(harm.index,1):save_crop(page,int(cid),'largest_false_preservation_increases',rank,clean,observed,mask,outputs,labels,stats)
    memory_log.append(memory_row('finalist_page_complete',page))
    print(f'Finalists {pi}/{len(pages)} {page}: RSS={rss_gb():.3f} GB')
    del z,clean,observed,mask,inv,outputs,route_sets,labels,stats;gc.collect()
components=pd.DataFrame(component_rows);components.to_csv(TABLES/'finalist_metrics_by_component.csv',index=False);pd.DataFrame(memory_log).to_csv(TABLES/'memory_telemetry.csv',index=False)


## Paired comparisons and manifest


In [ ]:
paired_rows=[]
for candidate in finalists:
    p=results[results.candidate.isin(['v78_baseline',candidate])].pivot(index='page',columns='candidate',values=['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved','removed_clean_print_fraction'])
    for metric in ['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved','removed_clean_print_fraction']:
        d=p[metric][candidate]-p[metric]['v78_baseline']
        paired_rows.append({'candidate':candidate,'metric':metric,'mean_delta':float(d.mean()),'median_delta':float(d.median()),'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
pd.DataFrame(paired_rows).to_csv(TABLES/'finalist_page_deltas.csv',index=False)

manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'version':'7.9-streamed-observed-ink-preservation-challenger-2',
 'notebook_name':'22_V7_9_Observed_Ink_Glyph_Preservation_Challenger.ipynb',
 'development_reference':str(V75),
 'development_pages':pages,
 'v7_8_independent_data_read':False,
 'notebook_09_read_or_modified':False,
 'baseline_equivalence_passed':bool(baseline_df.geometry_exact.all()),
 'candidate_modes':MODES,
 'candidate_configurations':len(configs)*len(MODES),
 'streamed_search':True,
 'full_grid_reconstruction_cache':False,
 'page_invariants_precomputed_once_per_page':True,
 'full_reconstruction_limited_to':['v78_baseline']+finalists,
 'selected_finalists':finalists,
 'selection_is_production_authorisation':False,
 'requires_new_structural_policy':True,
 'requires_new_untouched_independent_validation':True,
 'peak_sampled_rss_gb':float(pd.DataFrame(memory_log).rss_gb.max()),
 'outputs':{
  'candidate_summary':str(TABLES/'candidate_summary.csv'),
  'page_metrics':str(TABLES/'candidate_metrics_by_page.csv'),
  'finalist_components':str(TABLES/'finalist_metrics_by_component.csv'),
  'memory_telemetry':str(TABLES/'memory_telemetry.csv'),
  'contact_sheets':str(CONTACTS),
 }
}
(OUT/'v7_9_challenger_manifest.json').write_text(json.dumps(manifest,indent=2))
lines=[
 'V7.9 STREAMED OBSERVED-INK CHALLENGER',
 '',
 f'Development pages: {len(pages)}',
 f'Candidate configurations: {manifest["candidate_configurations"]}',
 f'Baseline equivalence: {manifest["baseline_equivalence_passed"]}',
 f'Selected finalists: {finalists}',
 f'Peak sampled RSS: {manifest["peak_sampled_rss_gb"]:.3f} GB',
 '',
 'Development evidence only. Notebook 09 unchanged. V7.8 independent outcomes not read.'
]
report=chr(10).join(lines)
(OUT/'executive_summary.txt').write_text(report)
print(report)
print('Output:',OUT)
